# Sheria Yangu — Know Your Rights Uganda
> *Sheria* means 'law' in Swahili. Every Ugandan citizen deserves to understand the documents that affect their life.

**Kaggle AI Agents Intensive Capstone — Agents for Good Track**

Sheria Yangu is a multi-agent AI system that helps Ugandan citizens understand legal documents in plain language. Upload or paste a contract, notice, or summons and the system tells you:
- What the document means
- What your rights are under Ugandan law
- What risks you face
- What options are available to you

**This system provides legal information, not legal advice.**

---

## Architecture
```
User Document
     |
     v
Orchestrator Agent
  |        |        |        |
Intake  Research  Analysis  Synthesis
Agent    Agent     Agent     Agent
  |        |        |        |
  +--------+--------+--------+
                |
         Citizen Report
```

**Key concepts demonstrated:**
- Multi-agent system (ADK pattern): Orchestrator + 4 specialist agents
- Custom MCP Server: Uganda statute knowledge base exposed as tools
- Antigravity model: Used in Analysis Agent for deep legal reasoning
- Security: Session-scoped only, no PII persists to disk

## Setup

In [ ]:
# Install dependencies
!pip install openai fastmcp fastapi uvicorn pypdf python-multipart pydantic python-dotenv -q

In [ ]:
# Clone the repository
!git clone https://github.com/crisomara/sheria_yangu.git
import sys, os
sys.path.insert(0, '/kaggle/working/sheria_yangu')
os.chdir('/kaggle/working/sheria_yangu')
print('Repository loaded.')

In [ ]:
# Load API key from Kaggle secrets
# Add your OpenRouter key under Add-ons > Secrets with name OPENROUTER_API_KEY
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ['OPENROUTER_API_KEY'] = secrets.get_secret('OPENROUTER_API_KEY')
print('API key loaded.')

In [ ]:
# Verify MCP server tools are available
from mcp_tools.server import list_acts, lookup_statutes

acts = list_acts()
print(f'Knowledge base loaded: {len(acts)} Acts')
for act in acts:
    print(f'  - {act}')

---
## Demo 1: Eviction Notice
A landlord has given a tenant 3 days to vacate with no reason stated. Is this lawful under Ugandan law?

In [ ]:
EVICTION_NOTICE = """
NOTICE TO VACATE

To: Mr. John Okello
    Plot 14, Nakawa Division, Kampala

From: Mr. Robert Ssemakula (Landlord)
Date: 1 July 2026

You are hereby required to vacate the above premises within THREE (3) DAYS
from the date of this notice.

Failure to vacate will result in your belongings being removed from the
premises and the locks being changed.

No reason is given for this notice.

Signed: R. Ssemakula
"""

In [ ]:
import asyncio
from utils.session import new_session
from agents.orchestrator import OrchestratorAgent

async def analyse(document: str):
    session_id = new_session()
    orchestrator = OrchestratorAgent(session_id=session_id)
    return await orchestrator.run(document_text=document)

result = await analyse(EVICTION_NOTICE)
print('Pipeline complete.')

In [ ]:
# Display the citizen report
from IPython.display import Markdown, display

def display_report(result):
    md = f"""## Sheria Yangu Report
**Document type:** {result['document_type']}

### What this document means
{result['summary']}

### Your rights under Ugandan law
"""
    for r in result['your_rights']:
        md += f"- {r}\n"

    md += "\n### Risks identified\n"
    for risk in result['risks']:
        severity_emoji = {'HIGH': '🔴', 'MEDIUM': '🟡', 'LOW': '🟢'}.get(risk['severity'], '⚪')
        md += f"""
**{severity_emoji} {risk['severity']} — {risk['plain_explanation']}**
- Document says: *{risk['what_document_says']}*
- Law says: *{risk['what_law_says']}*
- Legal basis: {risk['legal_basis']}
"""

    if result['deadlines']:
        md += "\n### Deadlines to be aware of\n"
        for d in result['deadlines']:
            urgency_emoji = {'IMMEDIATE': '🚨', 'SHORT_TERM': '⚠️', 'GENERAL': 'ℹ️'}.get(d['urgency'], 'ℹ️')
            md += f"- {urgency_emoji} **{d['description']}** — {d['date_mentioned']}\n"

    md += "\n### What you can do\n"
    for step in result['next_steps']:
        md += f"- {step}\n"

    md += "\n### Get legal help\n"
    for ref in result['legal_referrals']:
        contact = ref.get('phone', ref.get('url', ''))
        md += f"- **{ref['name']}** ({contact}) — {ref['notes']}\n"

    md += f"\n---\n> ⚠️ *{result['disclaimer']}*"
    display(Markdown(md))

display_report(result)

---
## Demo 2: Employment Contract
An employee is asked to sign a contract with a 5-day termination notice and only 10 days annual leave. What does Ugandan law say?

In [ ]:
EMPLOYMENT_CONTRACT = """
EMPLOYMENT CONTRACT

Employee: Ms. Grace Atim
Employer: Kampala Trading Company Ltd
Date: 1 July 2026

Section 7 - Termination:
The employer may terminate this contract at any time by giving the employee
FIVE (5) DAYS written notice, regardless of the employee's length of service.
The employee waives any right to claim unfair termination provided the
employer pays the 5-day notice pay.

Section 8 - Annual Leave:
The employee is entitled to 10 working days of annual leave per year.

Section 9 - Probation:
The employee shall serve a probation period of 12 months during which
the employer may terminate without notice or reason.
"""

result2 = await analyse(EMPLOYMENT_CONTRACT)
display_report(result2)

---
## Demo 3: Police Summons
A citizen receives a police summons. The document says they are not required to bring a lawyer. What are their rights?

In [ ]:
POLICE_SUMMONS = """
UGANDA POLICE FORCE
CENTRAL POLICE STATION, KAMPALA

SUMMONS TO ATTEND

To: Ms. Sarah Namukasa
    Wandegeya, Kampala

You are hereby summoned to appear at Central Police Station, Kampala
on 5 July 2026 at 9:00 AM to answer questions in connection with a
matter under investigation.

Failure to appear may result in your arrest.
You are not required to bring a lawyer.

Officer: D/Cpl James Opolot
Badge No: 4471
"""

result3 = await analyse(POLICE_SUMMONS)
display_report(result3)

---
## MCP Server Demo
Showing the custom MCP server tools working directly

In [ ]:
# Demo: MCP tool - list all Acts in the knowledge base
from mcp_tools.server import list_acts, lookup_statutes, get_section

print('Acts in the Sheria Yangu knowledge base:')
for act in list_acts():
    print(f'  {act}')

In [ ]:
# Demo: MCP tool - lookup statutes for eviction notice
statutes = lookup_statutes('Eviction Notice')
print(f'Found {len(statutes)} relevant provisions for an Eviction Notice:\n')
for s in statutes:
    print(f"  {s['act']} — {s['section']}: {s['title']}")

In [ ]:
# Demo: MCP tool - get a specific section
section = get_section('Employment Act', 'Section 58')
if section:
    print(f"{section['act']} — {section['section']}")
    print(f"Title: {section['title']}")
    print(f"\nText:\n{section['text']}")

---
## Security Design
Demonstrating that no user data persists after pipeline completion

In [ ]:
from utils.session import new_session, get_session, _sessions

# Show session is destroyed after pipeline runs
print(f'Active sessions after all pipeline runs: {len(_sessions)}')
print('All document content destroyed after processing.')
print('No PII persists to disk or database.')

---
*Sheria Yangu — Know Your Rights Uganda*

*Built for the Kaggle AI Agents Intensive Capstone — Agents for Good Track*

*This system provides legal information based on Ugandan law, not legal advice.*
*For advice specific to your situation, contact the Uganda Law Society: 0414-254848*